## Runtime Analysis

This is a notebook to evaluate the runtime of Parflex sparse and Dense with relation to FlexDTW

## Import nessecarry Functions

In [ ]:
## import alignment functions
import Parflex
import Sparse_Parflex
import FlexDTW
import numpy as np
from pathlib import Path
import pandas as pd
import librosa as lb
import time

import gc


In [16]:
OUTPUT_DIR = Path("/home/ijain/parflex/symphony_of_tears_features")
OUTPUT_DIR.mkdir(exist_ok=True)


In [17]:
def compute_and_save_chroma(audio_path):
    audio_path = Path(audio_path)
    y, sr = lb.core.load(str(audio_path), sr=None)
    feats = lb.feature.chroma_cqt(y=y, sr=sr, hop_length=512)
    feats = lb.util.normalize(feats, norm=2, axis=0)

    output_path = OUTPUT_DIR / f"{audio_path.stem}.npy"
    np.save(output_path, feats)
    return output_path

In [18]:


rec_1 = Path("/home/ijain/parflex/Symphony_of_Tears_(Manookian,_Jeff)/movement_1/PMLP118441-Armenia_performance.wav")
rec_2 = Path("/home/ijain/parflex/Symphony_of_Tears_(Manookian,_Jeff)/movement_1/PMLP118441-New_Disc_-01-_Track_01.wav")
# only do this if the file doesn't already exist
if not OUTPUT_DIR.exists():
    compute_and_save_chroma(rec_1)
    compute_and_save_chroma(rec_2)






In [19]:
N_TRIALS = 10
L_VALUES = [1000, 2000, 4000]
RESULTS_PATH = OUTPUT_DIR / "runtime_trials.csv"
SUMMARY_PATH = OUTPUT_DIR / "runtime_summary.csv"

trial_records = []


In [20]:
steps = np.array([[1, 1], [1, 2], [2, 1]], dtype=int)
weights = np.array([1.5, 3.0, 3.0], dtype=float)
beta = 0.1


In [21]:
P_VALUES = [100, 200, 500, 1000, 2000, 5000, 10000, 20000, 50000, 100000]

F1 = np.load(OUTPUT_DIR / "PMLP118441-Armenia_performance.npy")
F2 = np.load(OUTPUT_DIR / "PMLP118441-New_Disc_-01-_Track_01.npy")

max_p = min(F1.shape[1], F2.shape[1])

for current_p in P_VALUES:
    if current_p > max_p:
        print(f"Skipping P={current_p}: only {max_p} frames available")
        continue

    # Compute the PxP pairwise Euclidean cost matrix between first current_p frames.
    X = F1[:, :current_p].T
    Y = F2[:, :current_p].T
    x2 = np.sum(X * X, axis=1, keepdims=True)
    y2 = np.sum(Y * Y, axis=1, keepdims=True).T
    C = np.sqrt(np.maximum(x2 + y2 - 2.0 * (X @ Y.T), 0.0))

    for trial in range(1, N_TRIALS + 1):
        start = time.time()
        dist = FlexDTW.flexdtw(C, steps, weights)
        flexdtw_time = time.time() - start
        trial_records.append({
            "system": "FlexDTW",
            "P": current_p,
            "L": np.nan,
            "trial": trial,
            "runtime_sec": flexdtw_time,
            "distance": dist,
        })

        for l_val in L_VALUES:
            start = time.time()
            dist = Parflex.parflex(C, steps, weights, beta, l_val)
            dense_time = time.time() - start
            trial_records.append({
                "system": "ParFlexDTW_dense_serial",
                "P": current_p,
                "L": l_val,
                "trial": trial,
                "runtime_sec": dense_time,
                "distance": dist,
            })

            start = time.time()
            dist = Sparse_Parflex.parflex(C, steps, weights, beta, l_val)
            sparse_time = time.time() - start
            trial_records.append({
                "system": "ParFlexDTW_sparse_serial",
                "P": current_p,
                "L": l_val,
                "trial": trial,
                "runtime_sec": sparse_time,
                "distance": dist,
            })
            gc.collect()

    print(f"Completed P={current_p}")


runtime_trials_df = pd.DataFrame(trial_records)
runtime_trials_df.to_csv(RESULTS_PATH, index=False)

runtime_summary_df = (
    runtime_trials_df
    .groupby(["system", "P", "L"], dropna=False, as_index=False)
    .agg(
        avg_runtime_sec=("runtime_sec", "mean"),
        std_runtime_sec=("runtime_sec", "std"),
        n_trials=("runtime_sec", "count"),
    )
    .sort_values(["P", "system", "L"], na_position="first")
)
runtime_summary_df.to_csv(SUMMARY_PATH, index=False)

print(f"Saved trial runtimes to: {RESULTS_PATH}")
print(f"Saved average/stdev summary to: {SUMMARY_PATH}")


Completed P=100
Completed P=200
Completed P=500
Completed P=1000
Completed P=2000
Completed P=5000
Completed P=10000
Completed P=20000


SystemError: CPUDispatcher(<function flexdtw at 0x7fb567ba89d0>) returned a result with an exception set